In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

In [2]:
df = pd.read_csv('insurance.csv')

In [3]:
df.sample(5)

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
95,36,52.8,1.57,19.64,False,Indore,business_owner,Low
85,33,51.4,1.86,34.66,False,Chennai,private_job,Low
46,42,83.0,1.57,25.57,True,Kolkata,unemployed,High
44,59,77.0,1.60,50.00,True,Lucknow,private_job,Medium
50,55,66.7,1.88,25.23,True,Jaipur,private_job,Medium


In [4]:
df_feat = df.copy()

In [5]:
#Feature 1: BMI
df_feat["bmi"] = df_feat["weight"] / (df_feat["height"] ** 2)

In [6]:
# Feature 2: Age GRoup
def age_group(age):
    if age < 25:
        return "young"
    elif age < 45:
        return "adult"
    elif age < 60:
        return "middle_aged"
        return "senior"

In [7]:
df_feat["age_group"] = df_feat["age"].apply(age_group)

In [8]:
# Feature 3: Lifestyle Risk
def lifestyle_risk(row):
    if row["smoker"] and row["bmi"] > 30:
        return "high"
    elif row["smoker"] or row["bmi"] > 27:
        return "medium"
    else:
        return "low"

In [9]:
df_feat["lifestyle_risk"] = df_feat.apply(lifestyle_risk,axis=1)

In [10]:
tier_1_cities = ["Mumbai", "Delhi", "Bangalore", "Chennai", "Kolkata", "Hyderabad", "Pune"]
tier_2_cities = [
    "Jaipur", "Chandigarh", "Indore", "Lucknow", "Patna", "Ranchi", "Visakhapatnam", "Coimbatore",
    "Bhopal", "Nagpur", "Vadodara", "Surat", "Rajkot", "Jodhpur", "Raipur", "Amritsar", "Varanasi",
    "Agra", "Dehradun", "Mysore", "Jabalpur", "Guwahati", "Thiruvananthapuram", "Ludhiana", "Nashik",
    "Allahabad", "Udaipur", "Aurangabad", "Hubli", "Belgaum", "Salem", "Vijayawada", "Tiruchirappalli",
    "Bhavnagar", "Gwalior", "Dhanbad", "Bareilly", "Aligarh", "Gaya", "Kozhikode", "Warangal",
    "Kolhapur", "Bilaspur", "Jalandhar", "Noida", "Guntur", "Asansol", "Siliguri"
]

In [11]:
# Feature 4: City Tier
def city_tier(city):
    if city in tier_1_cities:
        return 1
    elif city in tier_2_cities:
      return 2
    else:
        return 3

In [12]:
df_feat["city_tier"] = df_feat["city"].apply(city_tier)

In [13]:
df_feat.drop(columns=['age', 'weight', 'height', 'smoker', 'city'])[['income_lpa', 'occupation', 'bmi', 'age_group', 'lifestyle_risk', 'city_tier', 'insurance_premium_category']].sample(5)

,income_lpa,occupation,bmi,age_group,lifestyle_risk,city_tier,insurance_premium_category
71,20.25,unemployed,16.513537,adult,low,2,Low
64,1.02,retired,37.179649,None,medium,2,High
34,0.68,retired,32.914286,None,high,3,High
60,49.94,unemployed,30.920912,adult,high,2,High
88,30.00,government_job,31.443698,middle_aged,medium,1,Low


In [14]:
# Select featurte and target 
x = df_feat[["bmi", "age_group", "lifestyle_risk", "income_lpa", "occupation"]]
y = df_feat["insurance_premium_category"]

In [15]:
x

,bmi,age_group,lifestyle_risk,income_lpa,occupation
0,49.227482,None,medium,2.92000,retired
1,30.189017,adult,medium,34.28000,freelancer
2,21.118382,adult,low,36.64000,freelancer
3,45.535900,young,high,3.34000,student
4,24.296875,None,medium,3.94000,retired
...,...,...,...,...,...
95,21.420747,adult,low,19.64000,business_owner
96,47.984483,adult,medium,34.01000,private_job
97,18.765432,middle_aged,low,44.86000,freelancer
98,30.521676,adult,medium,28.30000,business_owner


In [16]:
y

0       High
1        Low
2        Low
3     Medium
4       High
       ...  
95       Low
96       Low
97       Low
98       Low
99       Low
Name: insurance_premium_category, Length: 100, dtype: object

In [17]:
# Define categorical and numeric feature
categorical_features = ["age_group", "lifestyle_risk", "occupation", "city_tier"]
numeric_features = ["bmi", "income_lpa"]

In [18]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

In [19]:
# Create a pipeline with preprocessing and random forest classifier
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

In [20]:
print(x.columns)

Index(['bmi', 'age_group', 'lifestyle_risk', 'income_lpa', 'occupation'], dtype='object')


In [21]:
categorical_features = [col for col in categorical_features if col in x.columns]
numeric_features = [col for col in numeric_features if col in x.columns]

In [22]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

In [23]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,
    random_state=1
)

pipeline.fit(X_train, y_train)

ValueError: A given column is not a column of the dataframe

In [ ]:
preprocessor

In [25]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [26]:
categorical_features = ['age_group', 'lifestyle_risk', 'occupation']
numeric_features = ['bmi', 'income_lpa']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('num', 'passthrough', numeric_features)
    ]
)

In [27]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

In [28]:
X_train, X_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=1
)

In [29]:
pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [30]:
# Predict and evaluate
y_pred = pipeline.predict(X_test)
accuracy_score(y_test, y_pred)

0.65

In [34]:
X_test.sample(5)

,bmi,age_group,lifestyle_risk,income_lpa,occupation
52,47.344720,young,medium,2.960000,student
69,21.942857,middle_aged,low,6.034487,government_job
81,31.866055,adult,high,22.190000,freelancer
17,31.176471,None,medium,2.230000,retired
36,21.713266,None,low,0.530000,retired


In [38]:
import pickle
# save the trained pipeline using pickel 
pickel_model_path = "model.pkl"
with open(pickel_model_path,"wb") as f:
    pickle.dump(pipeline, f)